In [1]:
import pandas as pd
from utils import *

df = local.read.csv("unit_costs")
df

,Unit,Cost (Minerals / Gas),Total Cost per Unit,Count for 700+ Resources,Total Cost
0,marine,50 / 0,50,14,700
1,reaper,50 / 50,100,7,700
2,marauder,100 / 25,125,6,750
3,ghost,150 / 125,275,3,825
4,hellion,100 / 0,100,7,700
5,hellbat,100 / 0,100,7,700
6,siege tank,150 / 125,275,3,825
7,cyclone,150 / 100,250,3,750
8,thor,300 / 200,500,2,"1,000"
9,viking,150 / 75,225,4,900


In [52]:
import sc2reader
import pandas as pd
import numpy as np
import tqdm
from glob import glob
from utils import *
from collections import defaultdict

all_dfs = []
game_number = 0
for account in ['sample/account1', 'sample/account2', 'sample/account3', 'sample/account4']:
# for account in ['sample/account1']:
    base_project_folder = get_base_project_folder()
    files = glob(f'{base_project_folder}/data/input/my_data/{account}/*')
    for file in tqdm.tqdm(files):
        
        try:
            replay = sc2reader.load_replay(file, load_map=True)
        except Exception as e:
            print(f"Failed to load {file}: {e}")
            continue
        if replay.map_name in ['Ruby Rock LE', 'Emerald City CE', 'Reclamation LE', 'Fields of Death', 'Gemgarden LE', 'New Bed of Chaos LE', 'Rhoskallian LE', 'Rust Bucket LE', 'Sludge City', 'Undercurrent LE', 'Yellowjacket', 'Phantom Mode', '']:
            continue
        game_number += 1
        if replay.map_name not in ['valid_maps', "At Eternity's Edge LE", 'Blackrock LE', 'Fear and Faith LE', 'Rainfall LE', 'Sanctuary III LE', 'Lockdown LE', 'Washout LE', 'Rorschach LE', 'Old Sun Temple LE']:
            continue

        is_valid_release = replay.release_string >= '5.0.16'
        if not is_valid_release:
            continue
        
        player1 = 'nemo'
        player2 = None
        player1_won = None
        player1_race = 'Zerg'
        player2_race = None
        for player in replay.players:
            if player.name == 'nemo' or player.name == 'Kairo':
                assert player.play_race == 'Zerg'
                player1_won = player.result
            else:
                if player2 != None:
                    raise Exception(player.name, player2)
                player2 = player.name
                player2_race = player.play_race
        print(player1, player2, player1_race, player2_race)

        all_my_units_made = defaultdict(int)
        for event in replay.events:
            seconds = event.frame / 22.4
            minutes = int(seconds // 60)
            remaining_seconds = int(seconds % 60)
            if seconds < 1:
                continue
        
            try:
                if player2 in str(event):
                    continue
                if event.player == player2:
                    continue
            except:
                pass
            if event.name == "UnitBornEvent":
                all_my_units_made[event.unit_type_name] += 1
                # print(event.unit_type_name)

            if event.name == 'UnitInitEvent':
                if 'ultra' in str(event.unit).lower():
                    all_my_units_made['Ultralisk'] += 5
                # if 'spire' in str(event.unit).lower():
                #     all_my_units_made['Corruptor'] += 5
                #     all_my_units_made['Mutalisk'] += 5

            if event.name == "UnitTypeChangeEvent":
                all_my_units_made[event.unit_type_name] += 1
                
            if seconds > 10*60:
                break

        all_opponent_units_made = defaultdict(int)
        for event in replay.events:
            seconds = event.frame / 22.4
            minutes = int(seconds // 60)
            remaining_seconds = int(seconds % 60)
            if seconds < 1:
                continue

            is_good = False
            try:
                if player2 in str(event):
                    is_good = True
                if event.player == player2:
                    is_good = True
            except:
                pass
            if is_good == False:
                continue
            if event.name == "UnitBornEvent":
                all_opponent_units_made[event.unit_type_name] += 1

            if event.name == 'UnitInitEvent':
                if 'ultra' in str(event.unit).lower():
                    all_my_units_made['Ultralisk'] += 5
                    
            if event.name == "UnitTypeChangeEvent":
                all_my_units_made[event.unit_type_name] += 1

                
            if seconds > 10*60:
                break

              
        output_df = pd.DataFrame()
        # larva_born_times_df['larva_born_time'] = larva_born_times_data
        output_df['game_number'] = [game_number]
        output_df['opponent_name'] = [player2]
        output_df['opponent_race'] = [player2_race]
        output_df['game_length_seconds_max_600'] = [int(np.round(seconds,0))]
        output_df['main_player_won'] = [player1_won]
        for key, value in all_my_units_made.items():
            output_df["my_" + str(key)] = [value]
        for key, value in all_opponent_units_made.items():
            output_df["opponent_" + str(key)] = [value]        
        # output_df = larva_born_times_df[['game_number', 'opponent_race', 'game_length_seconds_max_600', 'main_player_won']]
        all_dfs.append(output_df)
output_df = pd.concat(all_dfs, axis='index', ignore_index=True)
local.write.csv(output_df, 'unit_comp_1_etl')


  3%|▎         | 1/33 [00:01<00:34,  1.08s/it]

nemo AcidCloud Zerg Terran


  6%|▌         | 2/33 [00:01<00:23,  1.34it/s]

nemo HORIZON Zerg Zerg


  9%|▉         | 3/33 [00:02<00:18,  1.65it/s]

...

nemo Aztk Zerg Zerg


 94%|█████████▍| 17/18 [00:08<00:00,  1.85it/s]

nemo Oogaly Zerg Zerg


100%|██████████| 18/18 [00:08<00:00,  2.10it/s]

nemo DarkStar Zerg Terran


In [51]:
1


1